# 42028: MLOps Part-1 - ClearML Dataset Versioning 

## Using Real Public Dataset: Cats vs Dogs (Small Public Subset)

**Learning Goals:**
- Download a small public dataset subset
- Organize into train/cats and train/dogs
- Create a ClearML versioned dataset
- Reproduce it anywhere
- Perform basic EDA and log it into ClearML 


## Step 0: Install ClearML and other libraries
Update this section if you need to intall additional libraries.

In [1]:
# Install required libraries (run once)
%pip install clearml tensorflow-datasets pillow -q
print("✅ clearml installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-timeseries 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
✅ clearml installed


In [ ]:
# Configure ClearML (run once per machine)
# Go to --> Settings --> Workspace --> Copy/Create New Credentials 
# --> Copy the Credentials
# --> Paste here when asked

!clearml-init

## Step 1: Download Small Public Subset (1%)
We use TensorFlow Datasets to fetch a tiny portion.

In [2]:
from clearml import Task

# Init ClearML task for dataset preparation
task_prep = Task.init(
    project_name="42028-ClearML-Dataset-Demo-W2",
    task_name="dataset-preparation",
    task_type=Task.TaskTypes.data_processing
)
print("✅ ClearML task started:", task_prep.id)
# Check on ClearML Web App to verify

ClearML Task: created new task id=ce120fcac52f43f4bd7e109a51797197


2026-02-26 02:35:42.425466: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ClearML results page: https://app.clear.ml/projects/ed6fa65b7d1940c7b1f7f66d4ca2941e/experiments/ce120fcac52f43f4bd7e109a51797197/output/log


Could not fetch GPU stats: NVML Shared Library Not Found


✅ ClearML task started: ce120fcac52f43f4bd7e109a51797197
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


In [ ]:
import tensorflow_datasets as tfds
import os
from PIL import Image

# Load only 1% for fast demo
dataset, info = tfds.load("cats_vs_dogs", split="train[:1%]", with_info=True)

base_path = "data/train"
cats_path = os.path.join(base_path, "cats")
dogs_path = os.path.join(base_path, "dogs")

os.makedirs(cats_path, exist_ok=True)
os.makedirs(dogs_path, exist_ok=True)

for i, example in enumerate(tfds.as_numpy(dataset)):
    image = example['image']
    label = example['label']
    class_name = info.features['label'].int2str(label)

    if class_name == 'cat':
        Image.fromarray(image).save(f"{cats_path}/cat_{i}.jpg")
    else:
        Image.fromarray(image).save(f"{dogs_path}/dog_{i}.jpg")

print("Public subset downloaded and saved locally.")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Corrupt JPEG data: 239 extraneous bytes before marker 0xd9
Corrupt JPEG data: 214 extraneous bytes before marker 0xd9
Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


In [ ]:
# Log dataset stats and close the preparation task
logger_prep = task_prep.get_logger()

import os
cats_count = len(os.listdir("data/train/cats"))
dogs_count = len(os.listdir("data/train/dogs"))

logger_prep.report_scalar(title="Dataset Counts", series="cats",  value=cats_count, iteration=0)
logger_prep.report_scalar(title="Dataset Counts", series="dogs",  value=dogs_count, iteration=0)
logger_prep.report_scalar(title="Dataset Counts", series="total", value=cats_count + dogs_count, iteration=0)

print(f"Cats: {cats_count} | Dogs: {dogs_count} | Total: {cats_count + dogs_count}")

task_prep.close()
print("✅ Dataset preparation task closed.")

## Step 2: Create ClearML Dataset (Version 1)
Core idea: create → upload → finalize

In [ ]:
from clearml import Dataset

dataset = Dataset.create(
    dataset_project="42028-ClearML-Dataset-Demo-W2",
    dataset_name="cats-dogs-public-v1"
)

dataset.add_files("data")
dataset.upload()
dataset.finalize()

print("Dataset ID:", dataset.id)

Now the dataset is **versioned and immutable**.

## Step 3: Reproduce the Dataset
Delete local folder if you want — you can always recover it.

In [ ]:
dataset = Dataset.get(dataset_name="cats-dogs-public-v1")

local_copy = dataset.get_local_copy()

print("Dataset downloaded to:", local_copy)

***Note***: 
* ClearML manages its own cache.
* You don’t choose the folder — ClearML handles reproducibility automatically.
* Hence, copy the dataset back to your current working directory

In [ ]:
# Copying the dataset from ClearML Cache to the current working directory
import shutil

shutil.copytree(local_copy, "./data", dirs_exist_ok=True)

# Key Teaching Points
- Public dataset → structured folder
- Dataset version created
- Version is immutable
- Reproducibility guaranteed

Next week: this dataset ID will be automatically logged in CNN training experiments.

## Step 4: Create dataset Version 2 

Add new images and create a child version.


In [ ]:
# Create Version 2 inheriting from v1
# Replace with your actual Dataset ID printed in Step 2
V1_DATASET_ID = "paste-your-dataset-id-here"

dataset_v2 = Dataset.create(
    dataset_project="42028-ClearML-Dataset-Demo-W2",
    dataset_name="cats-dogs-public-v2",
    parent_datasets=[V1_DATASET_ID]
)

# Use add_files instead of sync_folder to avoid empty-diff issues
dataset_v2.add_files("data")
dataset_v2.upload()
dataset_v2.finalize()

print("Version 2 created:", dataset_v2.id)


## Step 5: Split Dataset, Plot Distribution & Log to ClearML

In [ ]:
import os, shutil, random
import matplotlib.pyplot as plt
from PIL import Image
from clearml import Task

# Init ClearML task
task = Task.init(
    project_name="42028-ClearML-Dataset-Demo-W2",
    task_name="split-and-eda",
    task_type=Task.TaskTypes.data_processing,
    reuse_last_task_id=False  # always create a fresh task
)
logger = task.get_logger()

# --- Split dataset 80/20 ---
random.seed(42)
src_dir = "./data/train"
counts = {"train": {}, "test": {}}

for cls in ["cats", "dogs"]:
    images = os.listdir(f"{src_dir}/{cls}")
    random.shuffle(images)
    n = int(len(images) * 0.8)
    splits = {"train": images[:n], "test": images[n:]}

    for split, files in splits.items():
        dest = f"./split_dataset/{split}/{cls}"
        os.makedirs(dest, exist_ok=True)
        for fname in files:
            shutil.copy(f"{src_dir}/{cls}/{fname}", f"{dest}/{fname}")
        counts[split][cls] = len(files)
        print(f"{split}/{cls}: {len(files)} images")

# Log counts as scalars
for split in ["train", "test"]:
    for cls in ["cats", "dogs"]:
        logger.report_scalar(title="Split Counts", series=f"{split}-{cls}", value=counts[split][cls], iteration=0)

# --- Bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ["steelblue", "coral"]
for ax, split in zip(axes, ["train", "test"]):
    vals = list(counts[split].values())
    bars = ax.bar(["Cats", "Dogs"], vals, color=colors)
    ax.set_title(f"{split.capitalize()} Set")
    ax.set_ylabel("Images")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v), ha="center", fontweight="bold")
plt.suptitle("Class Distribution (80/20 split)")
plt.tight_layout()
plt.savefig("/tmp/distribution.png", dpi=120)
plt.show()

# Flush plot to ClearML
logger.report_image(title="Class Distribution", series="bar chart", local_path="/tmp/distribution.png", iteration=0)
logger.flush()
print("\n✅ Distribution logged to ClearML")


In [ ]:
# --- Show and log sample images ---
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for row, cls in enumerate(["cats", "dogs"]):
    folder = f"./split_dataset/train/{cls}"
    images = sorted(os.listdir(folder))[:4]
    for col, fname in enumerate(images):
        img = Image.open(f"{folder}/{fname}").resize((128, 128))
        axes[row][col].imshow(img)
        axes[row][col].axis("off")
        axes[row][col].set_title(cls.capitalize() if col == 0 else "")

plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.savefig("/tmp/samples.png", dpi=120)
plt.show()

# Log sample grid
logger.report_image(title="Sample Images", series="train grid", local_path="/tmp/samples.png", iteration=0)

# Log individual images per class
for cls in ["cats", "dogs"]:
    folder = f"./split_dataset/train/{cls}"
    for i, fname in enumerate(sorted(os.listdir(folder))[:4]):
        logger.report_image(
            title=f"Sample {cls.capitalize()}",
            series=f"img_{i+1}",
            local_path=f"{folder}/{fname}",
            iteration=0
        )

# Flush all pending logs before closing
logger.flush()

task.close()
print("✅ All data flushed and task closed.")
print(f"View at: https://app.clear.ml")
